# Audit fixed sequence splits locally or on Colab.



In [1]:
from pathlib import Path
import os
import subprocess
import sys
import json

# Colab mounts Drive; local runs discover the repo from the notebook directory.
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False

project_override = os.environ.get('UA_DETRAC_PROJECT')
candidates = [Path.cwd(), *Path.cwd().parents]
if project_override:
    candidates.insert(0, Path(project_override))
if IN_COLAB and not project_override:
    candidates.insert(0, Path('/content/drive/MyDrive/CS331/CS331-Vehicle-Tracking'))
ROOT_PATH = next(
    (path for path in candidates if (path / 'scripts/prepare_dataset.py').is_file()),
    None,
)
if ROOT_PATH is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')

sys.path.insert(0, str(ROOT_PATH))
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     str(ROOT_PATH / 'requirements.txt')],
    check=True,
)
LOCAL_RAW_PATH = Path(os.environ.get('UA_DETRAC_RAW', ROOT_PATH / 'data/raw'))
if IN_COLAB and 'UA_DETRAC_RAW' not in os.environ:
    downloaded_raw = Path('/content/ua_detrac/raw')
    if downloaded_raw.is_dir():
        LOCAL_RAW_PATH = downloaded_raw
DATA_RAW_PATH = LOCAL_RAW_PATH
WORK_ROOT = Path('/content/ua_detrac') if IN_COLAB else ROOT_PATH / 'outputs/ua_detrac'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET = Path(os.environ.get('UA_DETRAC_OUTPUT', WORK_ROOT / 'dataset_v1.1'))
INVENTORY_PATH = WORK_ROOT / 'raw_inventory.json'
import pandas as pd
import matplotlib.pyplot as plt

print('Project:', ROOT_PATH)
print('Raw:', DATA_RAW_PATH)
print('Dataset output:', DATASET)








Project: d:\ZB\Code\UIT\CS331\CS331-Vehicle-Tracking
Raw: d:\ZB\Code\UIT\CS331\CS331-Vehicle-Tracking\data\raw
Dataset output: d:\ZB\Code\UIT\CS331\CS331-Vehicle-Tracking\outputs\ua_detrac\dataset_v1.1


In [3]:
from src.data.split import load_splits
from src.data.utils import read_yaml
splits = load_splits(DATASET / 'metadata')
official = read_yaml(ROOT_PATH / 'configs/data/official_sequences.yaml')
manifest = json.loads((DATASET / 'metadata/manifest.json').read_text())
if manifest['scope'] == 'full':
    assert set(splits['test']) == set(official['test'])
    assert set(splits['train'] + splits['val']) == set(official['train'])
s = pd.read_csv(DATASET / 'metadata/sequences.csv')
a = pd.read_csv(DATASET / 'canonical/annotations.csv').merge(s[['sequence', 'split', 'weather']], on='sequence')
display(pd.read_csv(DATASET / 'metadata/dataset_stats.csv'))
display(pd.crosstab(a.split, a.class_name, normalize='index'))
display(pd.crosstab(s.split, s.weather, normalize='index'))
print('No sequence leakage. Inspect distribution imbalance before freezing a new dataset version.')



,split,num_sequences,num_frames,num_annotated_frames,num_unannotated_frames,annotation_coverage,num_objects,num_tracks,car_count,bus_count,...,median_objects_per_frame,mean_bbox_width,mean_bbox_height,mean_bbox_area,median_track_length,occlusion_count,occlusion_unknown,occlusion_region_count,truncation_count,truncation_unknown
0,train,48,67790,66811,979,0.985558,473719,5268,402630,22153,...,6.0,91.194291,62.169674,7806.831643,71.0,0,473719,122615,56081,0
1,val,12,16001,15274,727,0.954565,124562,684,101223,11498,...,6.0,96.575140,68.879219,8813.086026,111.0,0,124562,51605,18706,0
2,test,40,56340,56167,173,0.996929,675774,2337,548555,71785,...,10.0,103.391807,63.510665,8897.171667,166.0,0,675774,290682,90056,0


class_name,bus,car,others,van
split,,,,
test,0.106226,0.811743,0.025031,0.057000
train,0.046764,0.849934,0.006124,0.097178
val,0.092307,0.812631,0.006623,0.088438


weather
split


No sequence leakage. Inspect distribution imbalance before freezing a new dataset version.
